# ATM-Net++ Spine MRI Segmentation — Google Colab
**Target: Val Dice > 0.85 on Tesla T4 (16GB)**

## Before running:
1. Runtime → Change runtime type → **T4 GPU**
2. Upload your SPIDER dataset (images.zip + masks.zip)
3. Run all cells in order

## Dataset structure needed:
```
SPIDER/
  images/  ← 447 .mha files
  masks/   ← 447 .mha files  
  overview.csv
  radiological_gradings.csv
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 1: Check GPU
# ═══════════════════════════════════════════════════════════
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')
    print(f'VRAM:    {torch.cuda.get_device_properties(0).total_memory // 1024**3} GB')
else:
    print('WARNING: No GPU — go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 2: Install dependencies
# ═══════════════════════════════════════════════════════════
!pip install SimpleITK opencv-python-headless scipy scikit-learn pandas -q
print('✓ Dependencies installed')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3: Upload SPIDER dataset
# ═══════════════════════════════════════════════════════════
# OPTION A: Upload from your computer (drag-drop or file picker)
# Upload these files:
#   - images.zip   (all *.mha files from images/ folder)
#   - masks.zip    (all *.mha files from masks/ folder)
#   - overview.csv
#   - radiological_gradings.csv

import os
from google.colab import files

print('Upload your SPIDER dataset files:')
print('  Required: images.zip, masks.zip, overview.csv, radiological_gradings.csv')
print('  OR: if already on Google Drive, run CELL 3B instead')
print()
print('Uploading now...')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 3B: Mount Drive (if dataset is already on Google Drive)
# Skip this cell if you uploaded in Cell 3
# ═══════════════════════════════════════════════════════════
# from google.colab import drive
# drive.mount('/content/drive')
# !ls /content/drive/MyDrive/SPIDER/

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 4: Extract dataset and verify
# ═══════════════════════════════════════════════════════════
import os, zipfile, glob
from pathlib import Path

# Create dataset directory
DATA_ROOT = Path('/content/SPIDER')
DATA_ROOT.mkdir(exist_ok=True)
(DATA_ROOT/'images').mkdir(exist_ok=True)
(DATA_ROOT/'masks').mkdir(exist_ok=True)

# Extract zip files if they exist
for zip_name, dest_dir in [('images.zip', DATA_ROOT/'images'),
                             ('masks.zip',  DATA_ROOT/'masks')]:
    if Path(zip_name).exists():
        print(f'Extracting {zip_name}...', end=' ', flush=True)
        with zipfile.ZipFile(zip_name, 'r') as z:
            z.extractall(dest_dir)
        print('done')
    elif Path(f'/content/{zip_name}').exists():
        print(f'Extracting /content/{zip_name}...', end=' ', flush=True)
        with zipfile.ZipFile(f'/content/{zip_name}', 'r') as z:
            z.extractall(dest_dir)
        print('done')

# Move CSV files if they're in /content
for csv_file in ['overview.csv', 'radiological_gradings.csv']:
    src = Path(f'/content/{csv_file}')
    dst = DATA_ROOT / csv_file
    if src.exists() and not dst.exists():
        import shutil
        shutil.copy(src, dst)
        print(f'Copied {csv_file}')

# Handle nested directories (zip might have extracted to a subfolder)
for subdir in ['images', 'masks']:
    target = DATA_ROOT / subdir
    mha_files = list(target.glob('**/*.mha'))
    # If files are nested, flatten them
    for f in mha_files:
        if f.parent != target:
            f.rename(target / f.name)

# Verify
n_images = len(list((DATA_ROOT/'images').glob('*.mha')))
n_masks  = len(list((DATA_ROOT/'masks').glob('*.mha')))
has_csv  = (DATA_ROOT/'overview.csv').exists()

print()
print('=== Dataset Verification ===')
print(f'Images: {n_images} .mha files')
print(f'Masks:  {n_masks} .mha files')
print(f'CSV:    {"✓" if has_csv else "MISSING - upload overview.csv"}')

if n_images < 10:
    print()
    print('ERROR: Images not found! Check extraction:')
    print('Files in /content/SPIDER/images/:')
    !ls /content/SPIDER/images/ | head -20
    print()
    print('Files directly in /content/:')
    !ls /content/*.mha 2>/dev/null | head -5
    !ls /content/*.zip 2>/dev/null
else:
    print(f'\n✓ Dataset ready: {n_images} images, {n_masks} masks')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 5: Training Configuration
# ═══════════════════════════════════════════════════════════
import torch

# Paths
DATA_ROOT   = '/content/SPIDER'
OUTPUT_DIR  = '/content/outputs'   # Change to Drive path to persist: '/content/drive/MyDrive/ATM_Net_outputs'

# Model config — optimized for T4 16GB
IMG_SIZE    = 384    # 384x384 (4x more pixels than laptop 192x192)
BATCH_SIZE  = 8     # 8 x 384x384 fp16 on 16GB T4
ACCUM_STEPS = 1
EPOCHS      = 200
LR          = 2e-4
WEIGHT_DECAY= 5e-4
MAX_SPP     = 30    # 30 slices per patient
USE_AMP     = True  # fp16 — doubles speed on T4

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify GPU memory
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory // 1024**3
    if vram_gb < 12:
        print(f'WARNING: Only {vram_gb}GB VRAM. Reducing batch size...')
        BATCH_SIZE = 4
        IMG_SIZE   = 256

print(f'Config: {IMG_SIZE}x{IMG_SIZE}  BS={BATCH_SIZE}  AMP={USE_AMP}')
print(f'Data:   {DATA_ROOT}')
print(f'Output: {OUTPUT_DIR}')
print(f'GPU:    {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 6: Full Training Script
# ═══════════════════════════════════════════════════════════
import sys, os, time, warnings, json, random, gc
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import SimpleITK as sitk
import cv2
from pathlib import Path
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import GradScaler, autocast
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# ── Paths & Config ────────────────────────────────────────
IMAGES_DIR  = Path(DATA_ROOT) / 'images'
MASKS_DIR   = Path(DATA_ROOT) / 'masks'
OVERVIEW    = Path(DATA_ROOT) / 'overview.csv'
CKPT_BEST   = Path(OUTPUT_DIR) / 'best_model.pth'
CACHE_DIR   = Path('/content/cache')
CACHE_DIR.mkdir(exist_ok=True)
NUM_CLASSES = 19; SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED); random.seed(SEED)

SPIDER_TO_ATMNET = {**{i:i for i in range(1,9)}, 100:9, **{201+i:10+i for i in range(8)}}
CLASS_NAMES = {
    0:'background',1:'Vert-1(L)',2:'Vert-2',3:'Vert-3',4:'Vert-4',
    5:'Vert-5',6:'Vert-6',7:'Vert-7',8:'Vert-8(U)',9:'Sacrum',
    10:'IVD-1(L)',11:'IVD-2',12:'IVD-3',13:'IVD-4',14:'IVD-5',
    15:'IVD-6',16:'IVD-7',17:'IVD-8(U)',18:'Canal',
}
VERT_CLASSES = list(range(1,9)); IVD_CLASSES = list(range(10,18)); RARE_CLASSES=[7,8,16,17]
CLASS_WEIGHTS = torch.tensor([
    0.0, 1.0,1.0,1.0,1.5,2.0,3.0,6.0,12.0, 1.0,
    6.0,4.0,4.0,5.0,6.0,9.0,14.0,30.0, 0.0
]).float()

def remap(m):
    out=np.zeros_like(m,dtype=np.int32)
    for s,d in SPIDER_TO_ATMNET.items(): out[m==s]=d
    return out

def fg_ratio(m): return float((m>0).sum())/max(m.size,1)

# ── Data Loading ─────────────────────────────────────────
def build_cache(pids, split):
    cache_file = CACHE_DIR/f'{split}_{IMG_SIZE}.npz'
    if cache_file.exists():
        print(f'  {split}: loading cache...', end=' ', flush=True)
        t0=time.time(); d=np.load(cache_file)
        imgs,msks,rare=d['imgs'],d['msks'],d['rare'].tolist()
        print(f'{len(imgs)} slices ({time.time()-t0:.1f}s)')
        return imgs,msks,rare

    print(f'  {split}: processing {len(pids)} patients...')
    imgs,msks,rare_flags=[],[],[]
    t0=time.time(); skipped=0
    for i,pid in enumerate(pids):
        fn=f'{pid}_t2.mha'
        ip=IMAGES_DIR/fn; mp=MASKS_DIR/fn
        if not ip.exists() or not mp.exists():
            skipped+=1; continue
        try:
            iv=sitk.GetArrayFromImage(sitk.ReadImage(str(ip))).astype(np.float32)
            mv=sitk.GetArrayFromImage(sitk.ReadImage(str(mp))).astype(np.int32)
        except Exception as e:
            skipped+=1; continue
        n=iv.shape[0]; lo,hi=int(n*0.05),int(n*0.95)
        ranked=sorted(range(lo,hi),key=lambda s:fg_ratio(remap(mv[s])),reverse=True)[:MAX_SPP]
        for s in ranked:
            rm=remap(mv[s])
            if fg_ratio(rm)<0.005: continue
            p1,p99=np.percentile(iv[s],[0.5,99.5])
            img_n=np.clip((iv[s]-p1)/(p99-p1+1e-8),0,1).astype(np.float32)
            img_r=cv2.resize(img_n,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_LINEAR).astype(np.float16)
            msk_r=cv2.resize(rm.astype(np.float32),(IMG_SIZE,IMG_SIZE),
                             interpolation=cv2.INTER_NEAREST).astype(np.uint8)
            imgs.append(img_r); msks.append(np.clip(msk_r,0,NUM_CLASSES-1))
            has_rare=float(sum((rm==c).sum() for c in RARE_CLASSES))/max(rm.size,1)>0.0003
            rare_flags.append(1.0 if has_rare else 0.1)
        if (i+1)%20==0:
            print(f'    {i+1}/{len(pids)} patients, {len(imgs)} slices so far...')

    if len(imgs)==0:
        raise ValueError(f'No slices loaded! Check DATA_ROOT={DATA_ROOT}\n'
                         f'Images dir: {IMAGES_DIR} (exists={IMAGES_DIR.exists()})\n'
                         f'Skipped: {skipped}/{len(pids)} patients')

    print(f'  {split}: {len(imgs)} slices ({skipped} patients skipped) | {time.time()-t0:.0f}s')
    imgs_arr=np.stack(imgs).astype(np.float16)
    msks_arr=np.stack(msks).astype(np.uint8)
    rare_arr=np.array(rare_flags,dtype=np.float32)
    np.savez_compressed(cache_file,imgs=imgs_arr,msks=msks_arr,rare=rare_arr)
    return imgs_arr,msks_arr,rare_flags

# ── Augmentation ─────────────────────────────────────────
class Aug:
    def __call__(self,img,msk):
        if random.random()<0.5: img=np.fliplr(img).copy(); msk=np.fliplr(msk).copy()
        if random.random()<0.6:
            a=random.uniform(-20,20)
            M=cv2.getRotationMatrix2D((IMG_SIZE//2,IMG_SIZE//2),a,1.0)
            img=cv2.warpAffine(img,M,(IMG_SIZE,IMG_SIZE),flags=cv2.INTER_LINEAR,borderMode=cv2.BORDER_REFLECT)
            mf=cv2.warpAffine(msk.astype(np.float32),M,(IMG_SIZE,IMG_SIZE),flags=cv2.INTER_NEAREST,borderMode=cv2.BORDER_CONSTANT)
            msk=np.clip(mf.astype(np.int32),0,NUM_CLASSES-1)
        gamma=random.uniform(0.65,1.5)
        img=np.clip(np.power(img.astype(np.float32)+1e-8,gamma),0,1)
        img=np.clip(img*random.uniform(0.75,1.25)+random.uniform(-0.1,0.1),0,1)
        if random.random()<0.4: img=np.clip(img+np.random.normal(0,0.015,img.shape),0,1)
        return img.astype(np.float32),msk.astype(np.int64)

class DS(Dataset):
    def __init__(self,imgs,msks,aug=None): self.imgs=imgs;self.msks=msks;self.aug=aug
    def __len__(self): return len(self.imgs)
    def __getitem__(self,i):
        img=self.imgs[i].astype(np.float32);msk=self.msks[i].astype(np.int64)
        if self.aug: img,msk=self.aug(img,msk)
        return torch.from_numpy(img[None]).float(),torch.from_numpy(msk).long()

# ── Model ─────────────────────────────────────────────────
class CA(nn.Module):
    def __init__(self,ch,r=8):
        super().__init__();r=max(1,ch//r)
        self.avg=nn.AdaptiveAvgPool2d(1);self.max=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Flatten(),nn.Linear(ch,r),nn.ReLU(True),nn.Linear(r,ch),nn.Sigmoid())
    def forward(self,x):
        a=self.fc(self.avg(x))+self.fc(self.max(x))
        return x*a.clamp(0,1).view(x.shape[0],-1,1,1)

class SA(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv2d(2,1,7,padding=3,bias=False),nn.BatchNorm2d(1),nn.Sigmoid())
    def forward(self,x): return x*self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True)[0]],1))

class RB(nn.Module):
    def __init__(self,ch):
        super().__init__()
        self.net=nn.Sequential(nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch),nn.ReLU(True),
                               nn.Conv2d(ch,ch,3,1,1,bias=False),nn.BatchNorm2d(ch))
        self.ca=CA(ch);self.sa=SA();self.act=nn.ReLU(True)
    def forward(self,x): return self.act(self.sa(self.ca(self.net(x)))+x)

class Enc(nn.Module):
    def __init__(self,ci,co,drop=0.0):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv2d(ci,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True),
                                nn.Conv2d(co,co,3,1,1,bias=False),nn.BatchNorm2d(co),nn.ReLU(True))
        self.res=RB(co);self.drop=nn.Dropout2d(drop) if drop>0 else nn.Identity()
    def forward(self,x): return self.drop(self.res(self.conv(x)))

class ResUNet(nn.Module):
    def __init__(self,b=32,nc=NUM_CLASSES,drop=0.2):
        super().__init__()
        self.e1=Enc(1,b);self.e2=Enc(b,b*2,drop*0.3)
        self.e3=Enc(b*2,b*4,drop*0.6);self.e4=Enc(b*4,b*8,drop*0.8)
        self.bn=nn.Sequential(Enc(b*8,b*16,drop),nn.Dropout2d(drop));self.pool=nn.MaxPool2d(2)
        self.u4=nn.ConvTranspose2d(b*16,b*8,2,2);self.d4=Enc(b*16,b*8,drop*0.4)
        self.u3=nn.ConvTranspose2d(b*8,b*4,2,2);self.d3=Enc(b*8,b*4,drop*0.2)
        self.u2=nn.ConvTranspose2d(b*4,b*2,2,2);self.d2=Enc(b*4,b*2)
        self.u1=nn.ConvTranspose2d(b*2,b,2,2);self.d1=Enc(b*2,b)
        self.ds3=nn.Conv2d(b*4,nc,1);self.ds2=nn.Conv2d(b*2,nc,1);self.out=nn.Conv2d(b,nc,1)
        self.aux=nn.Sequential(nn.Conv2d(b,b,3,1,1,bias=False),nn.BatchNorm2d(b),nn.ReLU(True),nn.Conv2d(b,nc,1))
    def forward(self,x):
        sz=x.shape[2:]
        e1=self.e1(x);e2=self.e2(self.pool(e1));e3=self.e3(self.pool(e2));e4=self.e4(self.pool(e3))
        d=self.bn(self.pool(e4))
        d=self.d4(torch.cat([self.u4(d),e4],1))
        d=self.d3(torch.cat([self.u3(d),e3],1))
        o3=F.interpolate(self.ds3(d),sz,mode='bilinear',align_corners=False)
        d=self.d2(torch.cat([self.u2(d),e2],1))
        o2=F.interpolate(self.ds2(d),sz,mode='bilinear',align_corners=False)
        d=self.d1(torch.cat([self.u1(d),e1],1))
        return (self.out(d),o2,o3,self.aux(d)) if self.training else self.out(d)

# ── Losses ────────────────────────────────────────────────
def dice_w(logits,tgt,sm=1e-6):
    B,C,H,W=logits.shape
    soft=F.softmax(logits,1);oh=F.one_hot(tgt.clamp(0,C-1),C).permute(0,3,1,2).float()
    p=soft[:,1:].reshape(B,C-1,-1);t=oh[:,1:].reshape(B,C-1,-1)
    inter=(p*t).sum(-1);union=p.sum(-1)+t.sum(-1);mask=(t.sum(-1)>0).float()
    dice=(2*inter+sm)/(union+sm);w=CLASS_WEIGHTS[1:].to(logits.device).view(1,C-1)
    return 1.0-(dice*mask*w).sum()/(mask*w).sum().clamp(min=1)

def focal_l(logits,tgt,gamma=2.0):
    ce=F.cross_entropy(logits,tgt.clamp(0,NUM_CLASSES-1),reduction='none')
    return ((1-torch.exp(-ce))**gamma*ce).mean()

def boundary_l(logits,tgt):
    soft=F.softmax(logits,1)
    oh=F.one_hot(tgt.clamp(0,NUM_CLASSES-1),NUM_CLASSES).permute(0,3,1,2).float()
    pooled=F.max_pool2d(oh[:,1:],3,stride=1,padding=1)
    boundary=(pooled-oh[:,1:]).clamp(0,1)
    w=CLASS_WEIGHTS[1:].to(logits.device).view(1,-1,1,1)
    return (boundary*(1-soft[:,1:])*w).sum()/((boundary*w).sum()+1e-6)

def compound(logits,tgt):
    tc=tgt.clamp(0,NUM_CLASSES-1)
    return (F.cross_entropy(logits,tc,label_smoothing=0.03)
            +dice_w(logits,tc)+0.3*focal_l(logits,tc)+0.15*boundary_l(logits,tc))

def total_loss(outs,tgt):
    o1,o2,o3,aux=outs;main=compound(o1,tgt)+0.3*compound(o2,tgt)+0.15*compound(o3,tgt)
    tc=tgt.clamp(0,NUM_CLASSES-1)
    rare_mask=sum((tc==c).float() for c in RARE_CLASSES).clamp(0,1)
    aux_ce=F.cross_entropy(aux,tc,reduction='none')
    return main+0.3*(aux_ce*(1+4*rare_mask)).mean()

@torch.no_grad()
def batch_dice(logits,tgt):
    pred=logits.argmax(1).cpu().numpy();gt=tgt.cpu().numpy();sm=1e-6;D=defaultdict(list)
    for b in range(pred.shape[0]):
        for c in range(1,NUM_CLASSES):
            p=(pred[b]==c).astype(float).ravel();t=(gt[b]==c).astype(float).ravel()
            if t.sum()==0 and p.sum()==0: continue
            tp=(p*t).sum();fp=(p*(1-t)).sum();fn=((1-p)*t).sum()
            D[c].append((2*tp+sm)/(2*tp+fp+fn+sm))
    all_d=[v for vs in D.values() for v in vs]
    return D,float(np.mean(all_d)) if all_d else 0.0

def get_splits():
    df=pd.read_csv(OVERVIEW);tr,va=[];va=[],[];seen=set()
    for name in df['new_file_name'].tolist():
        if not name.endswith('_t2') or 'SPACE' in name: continue
        pid=name.replace('_t2','')
        if pid in seen: continue
        seen.add(pid)
        sub=df.loc[df['new_file_name']==name,'subset'].values
        (va if len(sub) and sub[0]=='validation' else tr).append(pid)
    return tr,va

print('✓ All functions defined')

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 7: START TRAINING
# ═══════════════════════════════════════════════════════════

def train():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('='*65)
    print('  ATM-Net++ Training on Google Colab')
    print(f'  GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
    print(f'  Config: {IMG_SIZE}x{IMG_SIZE}  BS={BATCH_SIZE}  AMP={USE_AMP}')
    print('='*65)

    tr_pids,va_pids=get_splits()
    print(f'\n  Patients: {len(tr_pids)} train | {len(va_pids)} val\n')

    # Verify files exist before cache
    found = sum(1 for p in tr_pids if (IMAGES_DIR/f'{p}_t2.mha').exists())
    print(f'  Found {found}/{len(tr_pids)} train patient files')
    if found == 0:
        print(f'  ERROR: No MHA files found in {IMAGES_DIR}')
        print(f'  Files in images dir:')
        files_found = list(IMAGES_DIR.glob('*'))[:10]
        for f in files_found: print(f'    {f.name}')
        return

    print('\n  Building data cache...')
    ti,tm,tr_rare=build_cache(tr_pids,'train')
    vi,vm,va_rare=build_cache(va_pids,'val')
    rare_cnt=sum(1 for r in tr_rare if r>0.5)
    ram_mb=(ti.nbytes+tm.nbytes+vi.nbytes+vm.nbytes)//1024**2
    print(f'\n  RAM used: ~{ram_mb} MB | Rare slices: {rare_cnt}/{len(tr_rare)}')

    aug=Aug()
    sampler=WeightedRandomSampler(torch.tensor(tr_rare),len(tr_rare),replacement=True)
    tr_dl=DataLoader(DS(ti,tm,aug),batch_size=BATCH_SIZE,sampler=sampler,num_workers=2,pin_memory=True)
    va_dl=DataLoader(DS(vi,vm),    batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)

    model=ResUNet(b=32,nc=NUM_CLASSES,drop=0.2).to(device)
    n_p=sum(p.numel() for p in model.parameters())
    print(f'\n  Model: {n_p/1e6:.2f}M params')
    print(f'  Batches: {len(tr_dl)} train | {len(va_dl)} val/epoch\n')

    # Load checkpoint if exists
    start_epoch=1; best=0.0
    if CKPT_BEST.exists():
        ckpt=torch.load(str(CKPT_BEST),map_location=device)
        model.load_state_dict(ckpt['model_state_dict'],strict=False)
        best=ckpt.get('best_dice',0.0)
        start_epoch=ckpt.get('epoch',0)+1
        print(f'  Resumed: epoch {ckpt.get("epoch")} | dice={best:.4f}\n')

    optim=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(optim,T_max=EPOCHS,eta_min=1e-6)
    scaler=GradScaler(enabled=USE_AMP)

    no_imp=0; t0tot=time.time()
    print(f'  {"Ep":>4}  {"TrDice":>8}  {"VaDice":>8}  {"Best":>8}  {"Gap":>6}  {"Sec":>5}')
    print('  '+'─'*50)

    for ep in range(start_epoch, EPOCHS+1):
        model.train(); tl_b,td_b=[],[]; t0=time.time()
        optim.zero_grad(set_to_none=True)

        for i,(imgs,msks) in enumerate(tr_dl):
            imgs,msks=imgs.to(device,non_blocking=True),msks.to(device,non_blocking=True)
            with autocast(enabled=USE_AMP):
                outs=model(imgs); loss=total_loss(outs,msks)/ACCUM_STEPS
            scaler.scale(loss).backward()
            if (i+1)%ACCUM_STEPS==0 or (i+1)==len(tr_dl):
                scaler.unscale_(optim)
                nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optim); scaler.update()
                optim.zero_grad(set_to_none=True)
            tl_b.append(loss.item()*ACCUM_STEPS)
            with torch.no_grad():
                _,d=batch_dice(outs[0],msks); td_b.append(d)

        sched.step()
        td=float(np.mean(td_b)); ep_s=time.time()-t0

        model.eval(); Dc=defaultdict(list); vl_b=[]
        with torch.no_grad():
            for imgs,msks in va_dl:
                imgs,msks=imgs.to(device,non_blocking=True),msks.to(device,non_blocking=True)
                with autocast(enabled=USE_AMP):
                    out=model(imgs); vl_b.append(compound(out,msks).item())
                D,_=batch_dice(out,msks)
                for c,v in D.items(): Dc[c].extend(v)

        all_v=[v for vs in Dc.values() for v in vs]
        vd=float(np.mean(all_v)) if all_v else 0.0; gap=td-vd

        if vd>best:
            best=vd; no_imp=0
            pc={CLASS_NAMES[c]:float(np.mean(v)) for c,v in Dc.items() if v}
            torch.save({'epoch':ep,'model_state_dict':model.state_dict(),
                        'best_dice':best,'per_class_dice':pc}, CKPT_BEST)
        else: no_imp+=1

        flag=' ★' if vd==best else ''
        print(f'  {ep:>4}  {td:>8.4f}  {vd:>8.4f}  {best:>8.4f}  {gap:>+6.3f}  {ep_s:>4.0f}s{flag}')

        if vd>=0.90: print('\n  🎯 TARGET ACHIEVED: Dice >= 0.90!'); break
        if vd>=0.85: print(f'  📈 Dice {vd:.4f} >= 0.85 — excellent!')
        if no_imp>=50: print(f'\n  Early stop'); break

    print(f'\n  Done: {(time.time()-t0tot)/3600:.2f}h | Best Dice: {best:.4f}')
    return best

# RUN TRAINING
best_dice = train()

In [ ]:
# ═══════════════════════════════════════════════════════════
# CELL 8: Final Evaluation
# ═══════════════════════════════════════════════════════════
if Path(OUTPUT_DIR+'/best_model.pth').exists():
    ckpt=torch.load(OUTPUT_DIR+'/best_model.pth',map_location='cpu')
    print(f'Best checkpoint: epoch {ckpt["epoch"]} | dice={ckpt["best_dice"]:.4f}')
    print()
    pc=ckpt.get('per_class_dice',{})
    vert_d=[v for k,v in pc.items() if 'Vert' in k]
    ivd_d =[v for k,v in pc.items() if 'IVD'  in k]
    all_d =list(pc.values())
    print(f'Vertebrae mean Dice: {np.mean(vert_d):.4f}')
    print(f'IVD mean Dice:       {np.mean(ivd_d):.4f}')
    print(f'Overall mean Dice:   {np.mean(all_d):.4f}')
    print()
    print('Per-class:')
    for name,d in sorted(pc.items(),key=lambda x:-x[1]):
        bar='█'*int(d*25); tag='★' if d>=0.90 else '✓' if d>=0.80 else '·' if d>=0.70 else ''
        print(f'  {name:<18} {d:.4f}  {bar} {tag}')
else:
    print('No checkpoint yet')